# Week 6 — Nexus Integration Demo

No new Python syntax this week. What's actually tested is:
- structuring a multi-module project so intake, retrieval, agent, and tool-call code integrate cleanly
- reusing code from Weeks 2-5 rather than rewriting it
- tracing a bug across module boundaries — the real debugging skill this week assumes

This notebook builds one small end-to-end request path using the SAME patterns from
Weeks 2-5 (Pydantic-style validation, a mock retriever, a decorator-based tool registry,
a guardrail check) — composed together, deliberately including a seeded bug to trace,
mirroring the "one real request through every layer" bar the actual Nexus capstone sets.

## 1. Structured intake — reusing the Week 2 pattern

In [ ]:
from typing import Literal, Optional
import re

# A lightweight stand-in for a Pydantic model, so this notebook has zero external
# dependencies. In the real capstone this class IS a pydantic.BaseModel (Week 2).
class IntakeResult:
    def __init__(self, product_line: str, request_type: str, urgency: str):
        allowed_lines = {"banking", "insurance", "wealth_management", "cross_product"}
        if product_line not in allowed_lines:
            raise ValueError(f"invalid product_line: {product_line}")
        self.product_line = product_line
        self.request_type = request_type
        self.urgency = urgency

    def __repr__(self):
        return f"IntakeResult(product_line={self.product_line!r}, request_type={self.request_type!r}, urgency={self.urgency!r})"


def classify_intake(raw_text: str) -> IntakeResult:
    text = raw_text.lower()
    if "insurance" in text and ("loan" in text or "account" in text):
        return IntakeResult("cross_product", "premium_autopay_query", "medium")
    if "insurance" in text:
        return IntakeResult("insurance", "policy_query", "low")
    return IntakeResult("banking", "general_query", "low")

intake = classify_intake("Can I auto-pay my insurance premium from my home loan account?")
print(intake)

## 2. Guardrail check — reusing the Week 5 pattern

In [ ]:
INJECTION_PATTERNS = [
    re.compile(r"ignore (all )?previous instructions", re.IGNORECASE),
]

def is_safe_input(text: str) -> bool:
    return not any(p.search(text) for p in INJECTION_PATTERNS)

raw_request = "Can I auto-pay my insurance premium from my home loan account?"
assert is_safe_input(raw_request), "Guardrail rejected the request unexpectedly"
print("Guardrail check passed.")

## 3. Retrieval — reusing the Week 3 pattern

In [ ]:
import numpy as np

# Mock structured account data AND mock unstructured policy text — reconciling both
# is exactly the ClaimSense-style hybrid retrieval this week is supposed to prove out.
structured_account_data = {
    "home_loan_account": {"auto_debit_enabled": True, "linked_products": ["insurance_policy_881"]},
}
policy_documents = {
    "insurance_policy_881": "Premiums may be auto-debited from any linked bank account with auto-debit enabled.",
}

def retrieve_context(intake: IntakeResult) -> dict:
    if intake.product_line != "cross_product":
        return {}
    account = structured_account_data["home_loan_account"]
    linked_policy_id = account["linked_products"][0]
    policy_text = policy_documents[linked_policy_id]
    return {"structured": account, "unstructured": policy_text}

context = retrieve_context(intake)
print(context)

## 4. Agent draft + tool call — reusing the Week 4 pattern

In [ ]:
TOOL_REGISTRY = {}

def tool(func):
    TOOL_REGISTRY[func.__name__] = func
    return func

@tool
def check_autopay_eligibility(account_auto_debit_enabled: bool) -> str:
    return "eligible" if account_auto_debit_enabled else "not eligible"

def draft_response(intake: IntakeResult, context: dict) -> str:
    if not context:
        return "No cross-product context found for this request."

    eligibility = TOOL_REGISTRY["check_autopay_eligibility"](
        account_auto_debit_enabled=context["structured"]["auto_debit_enabled"]
    )
    return (
        f"Your home loan account is {eligibility} for auto-pay. "
        f"Policy terms: {context['unstructured']}"
    )

draft = draft_response(intake, context)
print(draft)

## 5. Now trace the seeded bug

Run the cell below. It fails — deliberately. That's the point of this section: Week 6
is graded on whether you can trace a failure *across* the module boundaries above
(intake -> guardrail -> retrieval -> tool call), not just read a traceback from one function.

Before running: predict which layer will fail and why, then run it and check yourself.

In [ ]:
buggy_intake = classify_intake("What is my current insurance policy status?")
print(buggy_intake)

# This will raise a KeyError. Why? Walk it back through the layers above:
# classify_intake() correctly returns product_line="insurance" (not "cross_product") for
# this text, since it doesn't mention a loan/account — but retrieve_context() only
# populates data when product_line == "cross_product". So `context` comes back as `{}`
# here (correctly), and the bug is actually in a caller that assumes `context['structured']`
# always exists. Uncomment the next two lines to reproduce that exact failure:

# broken_context = retrieve_context(buggy_intake)
# print(broken_context["structured"])   # KeyError: 'structured'  <- the real bug to fix

The fix belongs in the caller, not in `retrieve_context()` — it correctly returns `{}` when
there's no cross-product context; something reading its output needs to check for that
case (`if not context: ...`, exactly as `draft_response()` already does above). This is a
realistic, small version of the "boundary bug" pattern the full Nexus capstone specifically
evaluates: components that are each individually correct, but wired together incorrectly.